# Klasifikasi DemogPairs Menggunakan ViT (Emosi) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-emotion.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_rf_vit-emotion_",
    results_path="results/demogpairs_rf_vit-emotion_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.8060185185185185
Precision : 0.806306424461563
Recall    : 0.8060185185185186
F1 Score  : 0.8056921603397407
               precision    recall  f1-score   support

Asian_Females     0.8883    0.8611    0.8745       360
  Asian_Males     0.7967    0.8056    0.8011       360
Black_Females     0.7569    0.7611    0.7590       360
  Black_Males     0.8025    0.8806    0.8397       360
White_Females     0.8229    0.8000    0.8113       360
  White_Males     0.7706    0.7278    0.7486       360

     accuracy                         0.8060      2160
    macro avg     0.8063    0.8060    0.8057      2160
 weighted avg     0.8063    0.8060    0.8057      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9587962962962963,0.8882521489971347,0.8611111111111112,0.8744710860366713,360
Asian_Males,0.9333333333333333,0.7967032967032966,0.8055555555555556,0.8011049723756904,360
Black_Females,0.9194444444444444,0.7569060773480663,0.7611111111111111,0.7590027700831025,360
Black_Males,0.9439814814814815,0.8025316455696202,0.8805555555555555,0.8397350993377483,360
White_Females,0.937962962962963,0.8228571428571428,0.8,0.811267605633803,360
White_Males,0.9185185185185185,0.7705882352941177,0.7277777777777777,0.7485714285714287,360


Confusion matrix saved: images\cm_rf_vit-emotion_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               310                 3                12                15                19                 1
         Asian_Males                 2               290                 5                17                22                24
       Black_Females                17                 2               274                36                 3                28
         Black_Males                 8                 7                25               317                 2                 1
       White_Females                10                25                 8                 5               288                24
         White_Males                 2                37                38                 5                16               262


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-emotion_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8060185185185185,0.8056921603397407,0.806306424461563,0.8060185185185186,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-emotion_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 3818.0,
 'days': 0,
 'hours': 1,
 'minutes': 3,
 'seconds': 38.0,
 'text': '0 hari 1 jam 3 menit 38.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 14653.0,
 'days': 0,
 'hours': 4,
 'minutes': 4,
 'seconds': 13.0,
 'text': '0 hari 4 jam 4 menit 13.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7946,0.8021,0.7922,0.7836,0.8102,0.7965,0.7963,0.7972,0.7965,10.166
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7969,0.8009,0.7922,0.783,0.8096,0.7965,0.7962,0.7971,0.7965,10.5254
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.794,0.8067,0.794,0.7841,0.8015,0.7961,0.7957,0.7962,0.7961,14.5778
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7975,0.8061,0.7917,0.7836,0.7992,0.7956,0.7952,0.7958,0.7956,14.2938
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.7448,0.7344,0.7378,0.7558,0.7483,0.7442,0.7445,0.7461,0.7442,6.4045
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.7448,0.7344,0.7378,0.7558,0.7483,0.7442,0.7445,0.7462,0.7442,6.6731
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.7257,0.7407,0.7315,0.7431,0.7517,0.7385,0.739,0.7412,0.7385,5.7867
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.7257,0.735,0.7274,0.7454,0.7517,0.737,0.7375,0.7397,0.737,6.7495
